# Qwen3.8 Threshold MoE — Full Training
Minimal runner: clone/install, mount Drive, validate once, train, resume. The hot 117+ GiB expert/optimizer work stays on Colab local disk for speed. Every configured checkpoint (and a clean interruption/completion) mirrors that work to Drive. CSV logs, graphs and resident checkpoints are written automatically.

In [ ]:
!rm -rf /content/My-works
!git clone -b moe-threshold-pretrain https://github.com/Logan17de/My-works.git
%cd /content/My-works/qwen38-threshold-moe
!pip install -r requirements.txt

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Validate layer-wise recomputation once

In [ ]:
!python train.py validate-layerwise --config configs/full_g4.json

## Full 80-layer / 500-expert training
`full_g4.json` uses `/content/qtm-work` for the live expert masters + Adafactor state and automatically fills available VRAM with as many expert layers as safely fit, keeping 20 GiB headroom. `--drive-root` is only the persistent destination. At `save_every_steps` (100 by default), clean interruption, or completion, the local work tree is synchronized to Drive. Training also creates `metrics.csv`, `training.png`, `routing.png`, live expert/cache/backend logs, and checkpoints automatically.

In [ ]:
!python train.py train --config configs/full_g4.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe

## Resume after a saved interruption
On a fresh Colab runtime this restores the last synchronized expert/optimizer work from Drive back to local disk, then resumes the matching checkpoint.

In [ ]:
!python train.py train --config configs/full_g4.json --drive-root /content/drive/MyDrive/qwen38-threshold-moe --resume latest